In [1]:
import sqlite3
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import time
import threading
import warnings

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

warnings.filterwarnings('ignore')

DB_PATH = 'audit_logs.db'
GAME_SPEED = 30.0

# Color palette
METAL_COLOR = '#4FC3F7'
ENERGY_COLOR = '#FFD54F'
SENDER_COLOR = '#EF5350'
RECEIVER_COLOR = '#66BB6A'
STORAGE_COLOR = '#FF6B6B'
TAX_COLOR = '#AB47BC'
UNTAXED_COLOR = '#66BB6A'
TAXED_COLOR = '#EF5350'
NEUTRAL_COLOR = '#78909C'
BG_COLOR = '#16213e'
PANEL_COLOR = '#1a1a2e'
GRID_COLOR = '#333355'

# Team color palette for distinguishing teams
TEAM_COLORS = [
    '#4FC3F7', '#66BB6A', '#FFD54F', '#AB47BC',
    '#FF7043', '#26C6DA', '#EC407A', '#9CCC65',
    '#7E57C2', '#29B6F6', '#FFA726', '#8D6E63'
]

def get_conn():
    return sqlite3.connect(DB_PATH, timeout=10)

def query_df(sql, params=None):
    conn = get_conn()
    try:
        return pd.read_sql_query(sql, conn, params=params)
    finally:
        conn.close()

def get_resource_color(resource):
    return METAL_COLOR if resource == 'metal' else ENERGY_COLOR

def get_team_color(team_id):
    return TEAM_COLORS[int(team_id) % len(TEAM_COLORS)]

# Team name cache: team_id -> player/AI name
_team_names = {}

def load_team_names():
    """Load team names from database if available."""
    global _team_names
    try:
        df = query_df("SELECT team_id, name FROM team_names ORDER BY team_id")
        if not df.empty:
            _team_names = dict(zip(df['team_id'], df['name']))
    except:
        _team_names = {}

def get_team_label(team_id):
    """Get display label for a team (player name if available, else Team N)."""
    team_id = int(team_id)
    if team_id in _team_names:
        return _team_names[team_id]
    return f'Team {team_id}'

def frame_to_time(frame):
    """Convert frame to game time in seconds."""
    return frame / GAME_SPEED

def format_game_time(seconds):
    """Format seconds as MM:SS."""
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins}:{secs:02d}"

print("✓ Libraries loaded | Plotly interactive charts enabled")


✓ Libraries loaded | Plotly interactive charts enabled


## 🎛️ Control Panel


In [2]:
# State
auto_refresh_enabled = False
refresh_interval = 5
selected_team = None  # None = ALL
selected_resource = 'metal'
selected_session = None  # None = latest
selected_source_path = None  # None = ALL, "PE" or "RE"
time_window = 100  # Show last N seconds of game time
use_game_time = True  # Toggle between game time and frame

# Widgets
session_selector = widgets.Dropdown(
    options=[('Latest Session', None)],
    value=None,
    description='📁 Session:',
    layout=widgets.Layout(width='220px')
)

source_path_selector = widgets.Dropdown(
    options=[('ALL Paths', None), ('PE (ProcessEconomy)', 'PE'), ('RE (ResourceExcess)', 'RE')],
    value=None,
    description='⚙️ Process:',
    layout=widgets.Layout(width='220px')
)

team_selector = widgets.RadioButtons(
    options=[('ALL Teams', None)],
    value=None,
    description='🎯 Team:',
    layout=widgets.Layout(width='180px')
)

resource_toggle = widgets.ToggleButtons(
    options=[('🔩 Metal', 'metal'), ('⚡ Energy', 'energy')],
    value='metal',
    description='Resource:'
)

refresh_toggle = widgets.ToggleButtons(
    options=[('▶ ON', True), ('⏸ OFF', False)],
    value=False,
    description='Auto-Refresh:'
)

time_toggle = widgets.ToggleButtons(
    options=[('⏱ Game Time', True), ('🎞 Frames', False)],
    value=True,
    description='X-Axis:'
)

interval_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=30,
    step=1,
    description='Interval (s):',
    continuous_update=False
)

window_slider = widgets.IntSlider(
    value=100,
    min=30,
    max=600,
    step=30,
    description='Time Window (s):',
    continuous_update=False
)

refresh_button = widgets.Button(
    description='🔄 Refresh Now',
    button_style='info'
)

status_label = widgets.HTML(value="<i>Ready</i>")
chart_output = widgets.Output()

def update_session_options():
    """Fetch available sessions from database."""
    try:
        sessions_df = query_df("""
            SELECT id, start_frame, end_frame, team_count, 
                   COALESCE(end_frame - start_frame, 0) as duration
            FROM game_sessions 
            ORDER BY id DESC
        """)
        if not sessions_df.empty:
            options = [('Latest Session', None)]
            for _, row in sessions_df.iterrows():
                duration_sec = row['duration'] / GAME_SPEED if row['duration'] else 0
                label = f"Session #{int(row['id'])} ({format_game_time(duration_sec)}, {int(row['team_count'] or 0)} teams)"
                options.append((label, int(row['id'])))
            session_selector.options = options
    except Exception as e:
        pass

def update_team_options():
    """Fetch available teams from database."""
    try:
        load_team_names()
        session_filter = ""
        if selected_session is not None:
            session_filter = f"WHERE session_id = {selected_session}"
        teams_df = query_df(f"SELECT DISTINCT team_id FROM eco_team_input {session_filter} ORDER BY team_id")
        if not teams_df.empty:
            options = [('ALL Teams', None)]
            options += [(get_team_label(int(t)), int(t)) for t in teams_df['team_id']]
            team_selector.options = options
    except Exception as e:
        print(f"Error loading teams: {e}")

update_session_options()
update_team_options()

# Event handlers
def on_session_change(change):
    global selected_session
    selected_session = change['new']
    update_team_options()

def on_source_path_change(change):
    global selected_source_path
    selected_source_path = change['new']

def on_team_change(change):
    global selected_team
    selected_team = change['new']

def on_resource_change(change):
    global selected_resource
    selected_resource = change['new']

def on_refresh_toggle(change):
    global auto_refresh_enabled
    auto_refresh_enabled = change['new']
    if auto_refresh_enabled:
        start_auto_refresh()

def on_time_toggle(change):
    global use_game_time
    use_game_time = change['new']

def on_interval_change(change):
    global refresh_interval
    refresh_interval = change['new']

def on_window_change(change):
    global time_window
    time_window = change['new']

def on_refresh_click(b):
    update_session_options()
    update_team_options()
    refresh_charts()

session_selector.observe(on_session_change, names='value')
source_path_selector.observe(on_source_path_change, names='value')
team_selector.observe(on_team_change, names='value')
resource_toggle.observe(on_resource_change, names='value')
refresh_toggle.observe(on_refresh_toggle, names='value')
time_toggle.observe(on_time_toggle, names='value')
interval_slider.observe(on_interval_change, names='value')
window_slider.observe(on_window_change, names='value')
refresh_button.on_click(on_refresh_click)

# Layout
control_box = widgets.VBox([
    widgets.HTML("<h3 style='margin:0;padding:10px 0;'>🎛️ Real-Time Economy Controls</h3>"),
    widgets.HBox([session_selector, source_path_selector, resource_toggle]),
    widgets.HBox([team_selector, time_toggle]),
    widgets.HBox([refresh_toggle, interval_slider, window_slider]),
    widgets.HBox([refresh_button, status_label])
], layout=widgets.Layout(
    padding='10px',
    border='1px solid #444',
    border_radius='5px'
))

display(control_box)


## 📊 Economy Chart


In [3]:
def calculate_income(df):
    """
    Derive income from frame-over-frame resource changes.
    income = Δcurrent + sent - received
    """
    if df.empty:
        return df
    
    df = df.sort_values(['team_id', 'frame']).copy()
    df['game_time'] = df['frame'] / GAME_SPEED
    df['prev_current'] = df.groupby('team_id')['current'].shift(1)
    df['income'] = df['current'] - df['prev_current'] + df['sent'] - df['received']
    
    df['income_smooth'] = df.groupby('team_id')['income'].transform(
        lambda x: x.rolling(window=30, min_periods=1).mean()
    )
    
    return df.dropna(subset=['income'])


def fetch_data(resource, time_min, time_max, team_id=None, session_id=None, source_path=None):
    """Fetch economy data for the specified time range (in seconds)."""
    
    frame_min = int(time_min * GAME_SPEED)
    frame_max = int(time_max * GAME_SPEED)
    
    filters = ["i.resource = ?", "i.frame BETWEEN ? AND ?"]
    params = [resource, frame_min, frame_max]
    
    if team_id is not None:
        filters.append(f"i.team_id = {team_id}")
    if session_id is not None:
        filters.append(f"i.session_id = {session_id}")
    if source_path is not None:
        filters.append(f"i.source_path = '{source_path}'")
    
    where_clause = " AND ".join(filters)
    
    main_df = query_df(f"""
        SELECT 
            i.frame,
            i.team_id,
            i.current as input_current,
            i.storage,
            COALESCE(i.source_path, '?') as source_path,
            COALESCE(o.current, i.current) as current,
            COALESCE(o.sent, 0) as sent,
            COALESCE(o.received, 0) as received
        FROM eco_team_input i
        LEFT JOIN eco_team_output o 
            ON i.frame = o.frame AND i.team_id = o.team_id AND i.resource = o.resource
        WHERE {where_clause}
        ORDER BY i.team_id, i.frame
    """, params)
    
    if not main_df.empty:
        main_df['game_time'] = main_df['frame'] / GAME_SPEED
    
    transfer_filters = ["resource = ?", "frame BETWEEN ? AND ?"]
    transfer_params = [resource, frame_min, frame_max]
    
    if team_id is not None:
        transfer_filters.append(f"(sender_team_id = {team_id} OR receiver_team_id = {team_id})")
    if session_id is not None:
        transfer_filters.append(f"session_id = {session_id}")
    if source_path is not None:
        transfer_filters.append(f"source_path = '{source_path}'")
    
    transfer_where = " AND ".join(transfer_filters)
    
    transfers_df = query_df(f"""
        SELECT frame, sender_team_id, receiver_team_id, amount, untaxed, taxed, 
               COALESCE(source_path, '?') as source_path
        FROM eco_transfer
        WHERE {transfer_where}
    """, transfer_params)
    
    if not transfers_df.empty:
        transfers_df['game_time'] = transfers_df['frame'] / GAME_SPEED
    
    return main_df, transfers_df


def create_economy_chart(resource, team_id=None, time_window=100, use_time=True, 
                         session_id=None, source_path=None):
    """
    Create interactive Plotly economy visualization with tooltips.
    
    - Line graph: Current resources per team with tooltips
    - Storage lines: Dotted red horizontal lines
    - Transfer bars: Stacked bars showing taxed/untaxed breakdown
    """
    
    session_filter = ""
    if session_id is not None:
        session_filter = f"WHERE session_id = {session_id}"
    
    max_frame_df = query_df(f"SELECT MAX(frame) as max_f FROM eco_team_input {session_filter}")
    if max_frame_df.empty or max_frame_df['max_f'].iloc[0] is None:
        return None, "No data available"
    
    frame_max = int(max_frame_df['max_f'].iloc[0])
    time_max = frame_max / GAME_SPEED
    time_min = max(0, time_max - time_window)
    
    main_df, transfers_df = fetch_data(resource, time_min, time_max, team_id, session_id, source_path)
    
    if main_df.empty:
        return None, f"No {resource} data in time range {format_game_time(time_min)}-{format_game_time(time_max)}"
    
    income_df = calculate_income(main_df)
    
    x_col = 'game_time' if use_time else 'frame'
    x_label = 'Game Time (seconds)' if use_time else 'Frame'
    
    fig = make_subplots(
        rows=3, cols=1,
        row_heights=[0.45, 0.30, 0.25],
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            f'📦 {resource.upper()} - Current Resources vs Storage',
            f'📈 {resource.upper()} - Income Rate',
            f'🔄 {resource.upper()} - Transfers (Taxed/Untaxed Breakdown)'
        )
    )
    
    teams = sorted(main_df['team_id'].unique())
    resource_emoji = '🔩' if resource == 'metal' else '⚡'
    
    # ===== ROW 1: Current Resources + Storage =====
    for team in teams:
        team_data = main_df[main_df['team_id'] == team]
        color = get_team_color(team)
        team_label = get_team_label(team)
        
        fig.add_trace(
            go.Scatter(
                x=team_data[x_col],
                y=team_data['current'],
                mode='lines',
                name=team_label,
                line=dict(color=color, width=2),
                hovertemplate=(
                    f'<b>{team_label}</b><br>'
                    'Time: %{x:.1f}s<br>'
                    'Current: %{y:,.0f}<br>'
                    '<extra></extra>'
                )
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=team_data[x_col],
                y=team_data['storage'],
                mode='lines',
                name=f'{team_label} Storage',
                line=dict(color=STORAGE_COLOR, width=1.5, dash='dot'),
                showlegend=False,
                hovertemplate=(
                    f'<b>{team_label} Storage</b><br>'
                    'Time: %{x:.1f}s<br>'
                    'Storage: %{y:,.0f}<br>'
                    '<extra></extra>'
                )
            ),
            row=1, col=1
        )
    
    # ===== ROW 2: Income Rate =====
    for team in teams:
        team_income = income_df[income_df['team_id'] == team]
        if not team_income.empty:
            color = get_team_color(team)
            team_label = get_team_label(team)
            fig.add_trace(
                go.Scatter(
                    x=team_income[x_col],
                    y=team_income['income_smooth'],
                    mode='lines',
                    name=f'{team_label} Income',
                    line=dict(color=color, width=1.5),
                    showlegend=False,
                    hovertemplate=(
                        f'<b>{team_label} Income</b><br>'
                        'Time: %{x:.1f}s<br>'
                        'Income: %{y:,.1f}/frame<br>'
                        '<extra></extra>'
                    )
                ),
                row=2, col=1
            )
    
    fig.add_hline(y=0, line=dict(color='#666', width=1), row=2, col=1)
    
    # ===== ROW 3: Transfer Stacked Bars (Taxed/Untaxed) =====
    if not transfers_df.empty:
        if team_id is not None:
            sent = transfers_df[transfers_df['sender_team_id'] == team_id].copy()
            received = transfers_df[transfers_df['receiver_team_id'] == team_id].copy()
            
            if not sent.empty:
                fig.add_trace(
                    go.Bar(
                        x=sent[x_col],
                        y=-sent['untaxed'],
                        name='Sent (Untaxed)',
                        marker_color=UNTAXED_COLOR,
                        opacity=0.8,
                        hovertemplate=(
                            '<b>Sent (Untaxed)</b><br>'
                            'Time: %{x:.1f}s<br>'
                            'Amount: %{y:,.1f}<br>'
                            '<extra></extra>'
                        )
                    ),
                    row=3, col=1
                )
                fig.add_trace(
                    go.Bar(
                        x=sent[x_col],
                        y=-sent['taxed'],
                        name='Sent (Taxed)',
                        marker_color=TAXED_COLOR,
                        opacity=0.8,
                        hovertemplate=(
                            '<b>Sent (Taxed)</b><br>'
                            'Time: %{x:.1f}s<br>'
                            'Amount: %{y:,.1f}<br>'
                            '<extra></extra>'
                        )
                    ),
                    row=3, col=1
                )
            
            if not received.empty:
                fig.add_trace(
                    go.Bar(
                        x=received[x_col],
                        y=received['untaxed'],
                        name='Received (Untaxed)',
                        marker_color=UNTAXED_COLOR,
                        opacity=0.6,
                        hovertemplate=(
                            '<b>Received (Untaxed)</b><br>'
                            'Time: %{x:.1f}s<br>'
                            'Amount: %{y:,.1f}<br>'
                            '<extra></extra>'
                        )
                    ),
                    row=3, col=1
                )
                fig.add_trace(
                    go.Bar(
                        x=received[x_col],
                        y=received['taxed'],
                        name='Received (Taxed)',
                        marker_color=TAXED_COLOR,
                        opacity=0.6,
                        hovertemplate=(
                            '<b>Received (Taxed)</b><br>'
                            'Time: %{x:.1f}s<br>'
                            'Amount: %{y:,.1f}<br>'
                            '<extra></extra>'
                        )
                    ),
                    row=3, col=1
                )
        else:
            agg = transfers_df.groupby(x_col).agg({
                'untaxed': 'sum',
                'taxed': 'sum',
                'amount': 'sum'
            }).reset_index()
            
            fig.add_trace(
                go.Bar(
                    x=agg[x_col],
                    y=agg['untaxed'],
                    name='Untaxed',
                    marker_color=UNTAXED_COLOR,
                    hovertemplate=(
                        '<b>Untaxed Transfers</b><br>'
                        'Time: %{x:.1f}s<br>'
                        'Amount: %{y:,.1f}<br>'
                        '<extra></extra>'
                    )
                ),
                row=3, col=1
            )
            fig.add_trace(
                go.Bar(
                    x=agg[x_col],
                    y=agg['taxed'],
                    name='Taxed',
                    marker_color=TAXED_COLOR,
                    hovertemplate=(
                        '<b>Taxed Transfers</b><br>'
                        'Time: %{x:.1f}s<br>'
                        'Amount: %{y:,.1f}<br>'
                        '<extra></extra>'
                    )
                ),
                row=3, col=1
            )
    
    fig.add_hline(y=0, line=dict(color='#666', width=1), row=3, col=1)
    
    team_str = f"Team {team_id}" if team_id is not None else "All Teams"
    time_range = f"{format_game_time(time_min)} - {format_game_time(time_max)}"
    path_str = f" | {source_path}" if source_path else ""
    session_str = f"Session #{session_id}" if session_id else "Latest"
    
    fig.update_layout(
        title=dict(
            text=f'{resource_emoji} Economy Dashboard | {session_str} | {team_str}{path_str} | {time_range}',
            font=dict(size=18, color='white')
        ),
        height=900,
        template='plotly_dark',
        paper_bgcolor=BG_COLOR,
        plot_bgcolor=PANEL_COLOR,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1,
            font=dict(size=10)
        ),
        barmode='stack',
        hovermode='x unified'
    )
    
    fig.update_xaxes(
        title_text=x_label,
        gridcolor=GRID_COLOR,
        row=3, col=1
    )
    fig.update_yaxes(gridcolor=GRID_COLOR)
    
    for i in range(1, 4):
        fig.update_xaxes(gridcolor=GRID_COLOR, row=i, col=1)
        fig.update_yaxes(gridcolor=GRID_COLOR, row=i, col=1)
    
    fig.update_yaxes(title_text='Amount', row=1, col=1)
    fig.update_yaxes(title_text='Income/frame', row=2, col=1)
    fig.update_yaxes(title_text='Transfer Amount', row=3, col=1)
    
    return fig, None


def refresh_charts():
    """Refresh the economy charts with current settings."""
    global selected_resource, selected_team, time_window, use_game_time
    global selected_session, selected_source_path
    
    load_team_names()
    
    with chart_output:
        clear_output(wait=True)
        
        try:
            fig, error = create_economy_chart(
                resource=selected_resource,
                team_id=selected_team,
                time_window=time_window,
                use_time=use_game_time,
                session_id=selected_session,
                source_path=selected_source_path
            )
            
            if fig:
                fig.show()
                status_label.value = f"<span style='color:#66BB6A'>✓ Updated {time.strftime('%H:%M:%S')}</span>"
            else:
                print(f"⚠️ {error}")
                status_label.value = f"<span style='color:#FFD54F'>⚠ {error}</span>"
                
        except Exception as e:
            import traceback
            print(f"❌ Error: {e}")
            traceback.print_exc()
            status_label.value = f"<span style='color:#EF5350'>❌ Error: {str(e)[:50]}</span>"


def auto_refresh_loop():
    """Background loop for auto-refresh."""
    global auto_refresh_enabled, refresh_interval
    
    while auto_refresh_enabled:
        refresh_charts()
        time.sleep(refresh_interval)


def start_auto_refresh():
    """Start the auto-refresh thread."""
    thread = threading.Thread(target=auto_refresh_loop, daemon=True)
    thread.start()
    status_label.value = "<span style='color:#4FC3F7'>🔄 Auto-refresh started</span>"


# Display chart output area
display(chart_output)

# Initial render
refresh_charts()


Output()

## 📋 Transfer Log Table

Recent transfer events in tabular form:


In [4]:
def show_recent_transfers(resource='metal', limit=20, team_id=None):
    """Display a table of recent transfer events with game time and source path."""
    
    team_filter = ""
    params = [resource, limit]
    
    if team_id is not None:
        team_filter = "AND (sender_team_id = ? OR receiver_team_id = ?)"
        params = [resource, team_id, team_id, limit]
    
    df = query_df(f"""
        SELECT 
            frame,
            ROUND(frame / 30.0, 1) as game_time,
            COALESCE(source_path, '?') as path,
            sender_team_id as sender,
            receiver_team_id as receiver,
            amount,
            untaxed,
            taxed,
            ROUND(amount - untaxed - taxed, 2) as tax_lost
        FROM eco_transfer
        WHERE resource = ? {team_filter}
        ORDER BY frame DESC
        LIMIT ?
    """, params)
    
    if df.empty:
        print(f"No {resource} transfers found")
        return
    
    df['time_fmt'] = df['game_time'].apply(lambda x: f"{int(x//60)}:{int(x%60):02d}")
    
    styled = df[['time_fmt', 'frame', 'path', 'sender', 'receiver', 'amount', 'untaxed', 'taxed', 'tax_lost']].style.format({
        'amount': '{:.1f}',
        'untaxed': '{:.1f}',
        'taxed': '{:.1f}',
        'tax_lost': '{:.1f}'
    }).background_gradient(
        subset=['amount'], cmap='Blues'
    ).background_gradient(
        subset=['taxed'], cmap='Reds'
    ).set_properties(**{
        'background-color': PANEL_COLOR,
        'color': 'white',
        'border-color': '#444'
    })
    
    display(HTML(f"<h4>📋 Recent {resource.title()} Transfers (last {limit})</h4>"))
    display(HTML("<p><small>Path: PE=ProcessEconomy, RE=ResourceExcess</small></p>"))
    display(styled)

# Show transfers for current selection
show_recent_transfers(selected_resource, 15, selected_team)


No metal transfers found


In [5]:
def show_economy_summary(resource='metal'):
    """Display summary statistics for the economy with game time."""
    
    latest_frame = query_df("SELECT MAX(frame) as max_f FROM eco_team_input")
    if latest_frame.empty or latest_frame['max_f'].iloc[0] is None:
        print("No data available")
        return
    
    max_frame = latest_frame['max_f'].iloc[0]
    game_time = max_frame / GAME_SPEED
    time_str = format_game_time(game_time)
    
    summary = query_df("""
        SELECT 
            i.team_id,
            i.current,
            i.storage,
            ROUND(i.current / i.storage * 100, 1) as fill_pct,
            i.share_slider,
            COALESCE(o.sent, 0) as last_sent,
            COALESCE(o.received, 0) as last_received
        FROM eco_team_input i
        LEFT JOIN eco_team_output o 
            ON i.frame = o.frame AND i.team_id = o.team_id AND i.resource = o.resource
        WHERE i.frame = (SELECT MAX(frame) FROM eco_team_input)
            AND i.resource = ?
        ORDER BY i.team_id
    """, (resource,))
    
    if summary.empty:
        print("No data available")
        return
    
    totals = query_df("""
        SELECT 
            SUM(sent) as total_sent,
            SUM(received) as total_received,
            SUM(sent) - SUM(received) as total_taxed
        FROM eco_team_output
        WHERE resource = ?
    """, (resource,))
    
    display(HTML(f"""
    <div style='background:{PANEL_COLOR};padding:15px;border-radius:5px;color:white;margin-bottom:10px;'>
        <h4 style='margin:0 0 10px 0;'>📊 {resource.title()} Economy Overview @ {time_str} (Frame {max_frame:,})</h4>
        <b>Total Sent:</b> {totals['total_sent'].iloc[0]:,.0f} | 
        <b>Total Received:</b> {totals['total_received'].iloc[0]:,.0f} | 
        <b>Total Taxed:</b> {totals['total_taxed'].iloc[0]:,.0f}
    </div>
    """))
    
    styled = summary.style.format({
        'current': '{:,.0f}',
        'storage': '{:,.0f}',
        'fill_pct': '{:.1f}%',
        'share_slider': '{:.2f}',
        'last_sent': '{:.1f}',
        'last_received': '{:.1f}'
    }).background_gradient(
        subset=['fill_pct'], cmap='RdYlGn', vmin=0, vmax=100
    ).set_properties(**{
        'background-color': PANEL_COLOR,
        'color': 'white'
    })
    
    display(styled)

show_economy_summary(selected_resource)


No data available


---

## Usage Tips

1. **Start the parser** in a terminal: `python parser.py --history` (or just `python parser.py` for live tailing)
2. **Enable auto-refresh** to see live updates as the game runs
3. **Filter by team** to focus on a specific player's economy
4. **Toggle resources** to switch between Metal and Energy views
5. **Adjust frame window** to see more or less history

### Legend

| Symbol | Meaning |
|--------|--------|
| Solid lines | Current resource level per team |
| Dotted red lines | Storage capacity |
| ▲ Green markers | Resources received |
| ▼ Red markers | Resources sent |
| Purple markers | Transfer events (all teams view) |
